In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy import log
from openpmd_viewer import OpenPMDTimeSeries
from scipy.constants import alpha, c, eV, m_e, pi

# useful constants
GeV = 1e9 * eV
gamma_em = 0.5772156649015  # Euler-Mascheroni

# electron energy
energy = 125 * GeV
gamma = energy / (m_e * c**2)


# virtual photons min energy
hw_min = 1e-12 * m_e * c**2

# min fractional energy of the virtual photon wrt electron energy
ymin = hw_min / energy

fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(5, 4), dpi=200)

#############
### WarpX ###
#############

series = OpenPMDTimeSeries("./diags/diag1/")
sampling_factor = 10000000
uz, w_wx = series.get_particle(["uz", "w"], species="virtual_photons", iteration=1)

w_primary_wx = series.get_particle(["w"], species="beam", iteration=1)

# fractional photon energy (photon energy / electron energy)
y_wx = uz * c / energy

# bins for the fractional photon energy
y = np.geomspace(ymin, 1, 201)

# number of virtual photons per electron obtained with WarpX
N_wx = np.sum(w_wx) / np.sum(w_primary_wx)
print(f"number of virtual photons per electron from WarpX = {N_wx}")

# spectrum of the virtual photons per electron
H, b = np.histogram(y_wx, bins=y, weights=w_wx)
db = np.diff(b)
b = 0.5 * (b[1:] + b[:-1])
dN_dy_wx = H / db / np.sum(w_primary_wx)

ax.plot(b, dN_dy_wx, label="WarpX", color="dodgerblue", lw=3, zorder=11)

##############
### Theory ###
##############

y = b
# spectrum of virtual photons for one electron
A = log(4) - 2 * gamma_em - 1
dN_dy = alpha / pi / y * (-2 * log(y) + A)
dN_dy[dN_dy < 0] = 0.0
ax.plot(y, dN_dy, color="red", lw=5, label="theory")

# number of virtual photons for one electron from theory
N = alpha / pi * (-A * log(ymin) + log(ymin) ** 2)
print(f"number of virtual photons per electron from theory = {N}")

ax.set_yscale("log")
ax.set_xscale("log")

ax.legend()
plt.show()

# do not consideer the last 4 bins
# not enough statistics
assert np.allclose(dN_dy[:-4], dN_dy_wx[:-4], rtol=1e-1)
assert np.isclose(N_wx, N, rtol=1e-5)